In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd

import os

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Hubs/grouped_hubs_for_filtering_29102025.csv', encoding='utf-8')

In [ ]:
df.head(1)

In [ ]:
df[df['group']==955]#[['TotalDemand']]

In [ ]:
df.columns

## Convert Line_Unique and Mode_Planned to a list of lines

In [ ]:
def split_lines(txt):
  txt = txt.replace("'","").replace("[","").replace("]","").replace("\"","").split(" ")

  return txt

In [ ]:
df['Line_Unique'] = df['Line_Unique'].apply(lambda x: split_lines(x))

In [ ]:
df['Mode_Planned'] = df['Mode_Planned'].apply(lambda x: split_lines(x))

## Convert node column to a list of nodes

In [ ]:
[int(x) for x in df['node'][0].replace("'","").replace("[","").replace("]","").replace("\"","").split(" ")]

In [ ]:
def node_list(txt):
  txt = [int(x) for x in txt.replace("'","").replace("[","").replace("]","").replace("\"","").replace(',',"").split(" ")]

  return txt

In [ ]:
df['node'] = df['node'].apply(lambda x: node_list(x))

## Convert h3_index column to a list of h3_indices

In [ ]:
def h3_index_list(txt):
  txt =  txt.replace("'","").replace("[","").replace("]","").replace("\"","").replace(',',"").split(" ")

  return txt

In [ ]:
df['h3_index'] = df['h3_index'].apply(lambda x: h3_index_list(x))

## Convert text columns: Model, Location, Area to a single value text

In [ ]:
def single_value(txt):
  txt = txt.replace("'","").replace("[","").replace("]","").replace("\"","").replace(",","").split(" ")[0]

  return txt

In [ ]:
df['Model'] = df['Model'].apply(lambda x: single_value(x))
df['location'] = df['location'].apply(lambda x: single_value(x))
df['area'] = df['area'].apply(lambda x: single_value(x))

## Result

In [ ]:
df.head(1)

In [ ]:
# Save result to reduce future run time
df.to_csv('/content/drive/MyDrive/Hubs/grouped_hubs_converted_ready_for_filtering_and_scoring_29102025.csv', encoding='utf-8')

# Categorize the different criteria columns (convert text to a numerical category)

### Region/Area

In [ ]:
df['area'].unique()

In [ ]:
# correct values spelling
def correct_area(txt):
  if txt == 'תל':
    return 'תל אביב'
  elif txt == 'באר':
    return 'באר שבע'
  else:
    return txt

In [ ]:
df['area'] = df['area'].apply(lambda x: correct_area(x))

In [ ]:
df['area'].unique()

In [ ]:
d = {k:1 for k in df['area'].unique() if k not in ['תל אביב','nan']}
d['תל אביב'] = 0
d['nan'] = 0

In [ ]:
def get_area_category(x) -> int:
  return d[x]

In [ ]:
df['Region_category'] = df['area'].apply(lambda x: get_area_category(x))

In [ ]:
df['Region_category'].head(5)

### Location

In [ ]:
df['location'].unique()

In [ ]:
def get_location_category(x) -> int:
  l = {'חיפה': 1,
       'צפון': 1,
       'דרום': 1,
       'טבעת': 2,
       'גלעין': 3,
       'nan': 1,
       '': 1}

  return l[x]

In [ ]:
df['Location_category'] = df['location'].apply(lambda x: get_location_category(x))

In [ ]:
df['Location_category'].unique()

### Region*Location

In [ ]:
df['RegionLocation'] = df['Region_category'] * df['Location_category']

### Mode score

In [ ]:
df.columns

In [ ]:
mode_weights = {
    'Funicular': 1,
    'Cable Line': 2,
    'BRT': 3,
    'LRT': 4,
    'Metro': 5,
    'Suburban Rail': 6,
    'Interurban Rail': 7,
    'HighSpeed Rail': 8
}

In [ ]:
def count_positive_mode_lines(row):
  count = 0
  mode_line_cols = ['BRT Lines', 'Cable Line Lines', 'Funicular Lines',
                    'HighSpeed Rail Lines', 'Interurban Rail Lines', 'LRT Lines',
                    'Metro Lines', 'Suburban Rail Lines']
  for col in mode_line_cols:
    if col in row.index and row[col] > 0:
      count += 1
  return count

In [ ]:
df['Num_Modes'] = df.apply(count_positive_mode_lines, axis=1)

In [ ]:
def get_mode_score(row):
  score = 0
  # for deminishing return
  alpha = 0.1
  for mode, weight in mode_weights.items():
    # Check if the mode exists in the Mode_Planned list
    if mode in row['Mode_Planned']:
      # Construct the column name
      column_name = f'{mode} Lines'
      # Check if the column exists in the DataFrame
      if column_name in row.index:
        score += row[column_name] * weight
  score = score * (1 + alpha * (row['Num_Modes'] - 1))
  return score

In [ ]:
# problem with Mode_Planned lists
# remove 'Rail', add ' Rail' to HighSpeed, Interurban, Suburban
# remove , from strings
def correct_mode_planned(x):
  if ('Rail' in x) or (' Rail' in x):
    x.remove('Rail')
  if 'HighSpeed' in x:
    x.append('HighSpeed Rail')
    x.remove('HighSpeed')
  if 'Interurban' in x:
    x.append('Interurban Rail')
    x.remove('Interurban')
  if 'Suburban' in x:
    x.append('Suburban Rail')
    x.remove('Suburban')

  for i in range(len(x)):
    x[i] = x[i].replace(',','')

  return x

In [ ]:
df['Mode_Planned'] = df['Mode_Planned'].apply(lambda x: correct_mode_planned(x))

In [ ]:
df['score'] = df.apply(get_mode_score, axis=1)

In [ ]:
df[df.index==8]['Mode_Planned'].values[0]

In [ ]:
df[df['Num_Modes']>1][['Mode_Planned','Num_Modes','BRT Lines', 'Cable Line Lines', 'Funicular Lines',
       'HighSpeed Rail Lines', 'Interurban Rail Lines', 'LRT Lines',
       'Metro Lines', 'Suburban Rail Lines','score']]

### Bus Terminal proximity
if bus terminal type is 'מסוף קטן' or 'מסוף בינוני' then value is 1 <br>
if type is 'מסוף גדול' or 'מתקן משולב' then value is 2
else: 0

In [ ]:
df['term_type'].unique()

In [ ]:
def bus_terminal(x) -> int:
  if x == 'חניון לילה':
    return 1
  elif x == 'מסוף קטן' or x == 'מסוף בינוני':
    return 2
  elif x == 'מסוף גדול' or x == 'מתקן משולב':
    return 3
  else:
    return 0

In [ ]:
df['bus_terminal'] = df['term_type'].apply(lambda x: bus_terminal(x))

In [ ]:
df['bus_terminal']

### Pop, Emp 2050
no action required at the moment <br>
scoring will be weight-based according to the ring <br>
will be done at the scoring phase (another notebook)

In [ ]:
df.info()

### Set Hub type

In [ ]:
def set_hub_type(df, cols):
  total_daily_demand = cols[0]
  service_types = cols[1]
  num_lines = len(cols[2]) # Get the length of the Line_Unique list

  # if has HighSpeed Rail or Interurban Rail, and Line Nunique is at least 3, Then it is a National type
  # if has Suburban Rail or Metro, and Line Nunique is at least 3, then it is a Regional type
  # if has BRT or LRT, Line Nunique is a least 3 and demand is greater then 10000, then it is a Local type
  # if has any Rail but Line Nunique is up to 2 or has less than 10000 daily demand, than it is Train Station
  # else - not hub
  if (('Interurban Rail' in service_types) or ('HighSpeed Rail' in service_types)) and (num_lines >= 3) and (total_daily_demand>=50000):
    return 'National'
  elif (('Suburban Rail' in service_types) or ('Metro' in service_types) or ('Interurban Rail' in service_types) or ('HighSpeed Rail' in service_types)) and (num_lines >= 3):
    return 'Regional'
  elif (('BRT' in service_types) or ('LRT' in service_types)) and (num_lines >= 3) and (total_daily_demand >= 1000):
    return 'Local'
  elif (('Interurban Rail' in service_types) or ('HighSpeed Rail' in service_types) or ('Suburban Rail' in service_types)) and (num_lines <= 2):
    return 'Train Station'
  else:
    return 'Not Hub'

In [ ]:
df.columns

In [ ]:
df['HubType'] = df[['TotalDemand', 'Mode_Planned', 'Line_Unique']].apply(lambda x: set_hub_type(df, x), axis=1)

In [ ]:
df.head()

In [ ]:
df.to_csv('/content/drive/MyDrive/Hubs/grouped_hubs_ready_for_scoring_29102025.csv', encoding='utf-8')